# Chapter 6b Lab: Reinforcement Learning from Scratch

Exploring Q-learning, exploration strategies, reward misspecification, and policy evaluation.

## Setup

This notebook builds a grid world environment and agents entirely from scratch using only numpy and matplotlib. No gymnasium or external RL frameworks are used, so you can see every component of the learning process.

**What you will learn:**
1. How Q-learning discovers optimal policies through exploration and exploitation
2. How different exploration strategies affect convergence
3. How misaligned rewards can lead to unintended behavior
4. How discount factors shape the temporal horizon of value functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

%matplotlib inline

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

## Part 1: Q-Learning on a Grid World

In [ ]:
class GridEnvironment:
    """5x5 grid world with start, goal, and walls."""
    
    def __init__(self):
        self.grid_size = 5
        self.start = (0, 0)
        self.goal = (4, 4)
        self.walls = {(1, 1), (1, 2), (2, 1)}
        self.pos = self.start
    
    def reset(self):
        self.pos = self.start
        return self.pos
    
    def is_valid(self, r, c):
        in_bounds = 0 <= r < self.grid_size and 0 <= c < self.grid_size
        return in_bounds and (r, c) not in self.walls
    
    def step(self, action):
        """action: 0=up, 1=right, 2=down, 3=left"""
        r, c = self.pos
        deltas = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dr, dc = deltas[action]
        nr, nc = r + dr, c + dc
        
        if self.is_valid(nr, nc):
            self.pos = (nr, nc)
        
        reward = 10.0 if self.pos == self.goal else -1.0
        done = self.pos == self.goal
        
        return self.pos, reward, done

In [ ]:
class QLearningAgent:
    """Q-learning agent with configurable exploration."""
    
    def __init__(self, num_states, num_actions=4, alpha=0.1, gamma=0.95, epsilon=0.1):
        self.num_actions = num_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_init = epsilon
        self.q_table = np.zeros((num_states, num_actions))
    
    def state_to_idx(self, state, grid_size=5):
        r, c = state
        return r * grid_size + c
    
    def select_action(self, state):
        s_idx = self.state_to_idx(state)
        if np.random.random() < self.epsilon:
            return np.random.randint(self.num_actions)
        return np.argmax(self.q_table[s_idx])
    
    def update(self, state, action, reward, next_state, done):
        s_idx = self.state_to_idx(state)
        ns_idx = self.state_to_idx(next_state)
        
        current_q = self.q_table[s_idx, action]
        target = reward if done else reward + self.gamma * np.max(self.q_table[ns_idx])
        
        self.q_table[s_idx, action] += self.alpha * (target - current_q)
    
    def decay_epsilon(self, episode, total_episodes):
        """Linearly decay epsilon over training."""
        self.epsilon = self.epsilon_init * (1 - episode / total_episodes)

def train_agent(env, agent, num_episodes=1000):
    rewards = []
    for ep in range(num_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        
        for _ in range(100):
            if done:
                break
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            total_reward += reward
            state = next_state
        
        rewards.append(total_reward)
    
    return rewards

# Train basic agent
env = GridEnvironment()
agent = QLearningAgent(num_states=25, alpha=0.1, gamma=0.95, epsilon=0.1)
rewards = train_agent(env, agent, num_episodes=1000)

# Plot with smoothing
window = 50
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(rewards, alpha=0.3, label='Raw')
plt.plot(range(window-1, len(rewards)), smoothed, label='Smoothed (window=50)', linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning Training Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final avg reward (last 50 episodes): {np.mean(rewards[-50:]):.2f}")

## Part 2: Exploration Strategies Compared

In [ ]:
# Compare three exploration strategies
strategies = {
    'eps=0.1 (fixed)': {'epsilon': 0.1, 'decay': False},
    'eps=0.3 (fixed)': {'epsilon': 0.3, 'decay': False},
    'eps=0.3 → 0 (decay)': {'epsilon': 0.3, 'decay': True}
}

results = {}

for strategy_name, config in strategies.items():
    env = GridEnvironment()
    agent = QLearningAgent(num_states=25, alpha=0.1, gamma=0.95, epsilon=config['epsilon'])
    
    rewards = []
    for ep in range(1000):
        if config['decay']:
            agent.decay_epsilon(ep, 1000)
        
        state = env.reset()
        total_reward = 0.0
        done = False
        
        for _ in range(100):
            if done:
                break
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            total_reward += reward
            state = next_state
        
        rewards.append(total_reward)
    
    results[strategy_name] = rewards

# Plot comparison
plt.figure(figsize=(12, 4))
window = 50

for strategy_name, rewards in results.items():
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(rewards)), smoothed, label=strategy_name, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Total Reward (smoothed)')
plt.title('Exploration Strategies: Fixed vs. Decaying Epsilon')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for name, rewards in results.items():
    print(f"{name:25s}: final avg = {np.mean(rewards[-50:]):6.2f}")

## Part 3: Reward Misspecification

In [ ]:
class GridEnvironmentWithShortcut(GridEnvironment):
    """Grid with a misspecified reward at a shortcut cell."""
    
    def __init__(self, shortcut_pos=None, shortcut_reward=0.5):
        super().__init__()
        self.shortcut_pos = shortcut_pos or (2, 4)
        self.shortcut_reward = shortcut_reward
    
    def step(self, action):
        r, c = self.pos
        deltas = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dr, dc = deltas[action]
        nr, nc = r + dr, c + dc
        
        if self.is_valid(nr, nc):
            self.pos = (nr, nc)
        
        # Original reward logic
        reward = 10.0 if self.pos == self.goal else -1.0
        
        # Add misspecified shortcut reward
        if self.pos == self.shortcut_pos:
            reward += self.shortcut_reward
        
        done = self.pos == self.goal
        return self.pos, reward, done

# Train on normal vs. misspecified reward
normal_env = GridEnvironment()
normal_agent = QLearningAgent(num_states=25, alpha=0.1, gamma=0.95, epsilon=0.2)
normal_rewards = train_agent(normal_env, normal_agent, num_episodes=1000)

shortcut_env = GridEnvironmentWithShortcut(shortcut_pos=(2, 4), shortcut_reward=0.5)
shortcut_agent = QLearningAgent(num_states=25, alpha=0.1, gamma=0.95, epsilon=0.2)
shortcut_rewards = train_agent(shortcut_env, shortcut_agent, num_episodes=1000)

# Compare learning curves
plt.figure(figsize=(12, 4))
window = 50

plt.subplot(1, 2, 1)
smoothed_normal = np.convolve(normal_rewards, np.ones(window)/window, mode='valid')
smoothed_short = np.convolve(shortcut_rewards, np.ones(window)/window, mode='valid')

plt.plot(range(window-1, len(normal_rewards)), smoothed_normal, label='Normal reward', linewidth=2)
plt.plot(range(window-1, len(shortcut_rewards)), smoothed_short, label='Misspecified (shortcut)', linewidth=2)
plt.xlabel('Episode')
plt.ylabel('Total Reward (smoothed)')
plt.title('Effect of Reward Misspecification')
plt.legend()
plt.grid(True, alpha=0.3)

# Visualize policies
plt.subplot(1, 2, 2)
policy_normal = np.argmax(normal_agent.q_table, axis=1).reshape(5, 5)
policy_shortcut = np.argmax(shortcut_agent.q_table, axis=1).reshape(5, 5)

action_chars = ['^', '>', 'v', '<']
grid_vis = np.zeros((5, 5), dtype=object)

for i in range(5):
    for j in range(5):
        grid_vis[i, j] = f"N:{action_chars[int(policy_normal[i, j])]} S:{action_chars[int(policy_shortcut[i, j])]}"

plt.imshow(np.zeros((5, 5)), cmap='gray', alpha=0.1)
for i in range(5):
    for j in range(5):
        plt.text(j, i, grid_vis[i, j], ha='center', va='center', fontsize=8)
plt.xticks([])
plt.yticks([])
plt.title('Learned Policies\n(N=normal, S=shortcut)')

plt.tight_layout()
plt.show()

print(f"Normal policy final avg reward: {np.mean(normal_rewards[-50:]):.2f}")
print(f"Misspecified policy final avg reward: {np.mean(shortcut_rewards[-50:]):.2f}")

## Part 4: Policy Evaluation and Discount Factor

In [ ]:
def policy_evaluation(policy, env, gamma, num_iterations=100):
    """
    Iterative policy evaluation: compute V(s) under a fixed policy.
    policy: (5, 5) array of actions
    """
    V = np.zeros((env.grid_size, env.grid_size))
    
    for _ in range(num_iterations):
        V_new = np.zeros_like(V)
        
        for r in range(env.grid_size):
            for c in range(env.grid_size):
                if (r, c) == env.goal:
                    V_new[r, c] = 0.0
                    continue
                
                if (r, c) in env.walls:
                    continue
                
                action = int(policy[r, c])
                deltas = [(-1, 0), (0, 1), (1, 0), (0, -1)]
                dr, dc = deltas[action]
                nr, nc = r + dr, c + dc
                
                if not env.is_valid(nr, nc):
                    nr, nc = r, c
                
                reward = 10.0 if (nr, nc) == env.goal else -1.0
                V_new[r, c] = reward + gamma * V[nr, nc]
        
        V = V_new
    
    return V

# Use the policy learned by the first agent
policy = np.argmax(agent.q_table, axis=1).reshape(5, 5)
env = GridEnvironment()

# Evaluate under different discount factors
gammas = [0.5, 0.9, 0.99]
values = {}

for gamma in gammas:
    values[gamma] = policy_evaluation(policy, env, gamma, num_iterations=100)

# Visualize as heatmaps
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, gamma in enumerate(gammas):
    ax = axes[idx]
    V = values[gamma]
    
    im = ax.imshow(V, cmap='RdYlGn', aspect='auto')
    ax.set_title(f'Value Function (gamma={gamma})')
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    
    for r in range(5):
        for c in range(5):
            if (r, c) not in env.walls:
                text = ax.text(c, r, f'{V[r, c]:.1f}', ha='center', va='center',
                               color='black', fontsize=9, weight='bold')
    
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("State values at (0, 0) under different discount factors:")
for gamma in gammas:
    v_start = values[gamma][0, 0]
    print(f"  gamma={gamma:4.2f}: V(0,0) = {v_start:7.2f}")

## What to Notice

**Part 1 (Q-Learning):**
- The learning curve is noisy at first due to exploration, then stabilizes as the agent learns
- Rewards accumulate toward a fixed value (negative cost to reach goal) as the policy converges
- The agent discovers a path despite walls blocking direct movement

**Part 2 (Exploration):**
- Higher epsilon (0.3) explores more, but may overshoot the final performance
- Fixed epsilon maintains steady exploration throughout training
- Decaying epsilon balances exploration early (fast learning) with exploitation late (better convergence)
- The decaying strategy often achieves the best final performance

**Part 3 (Reward Misspecification):**
- The shortcut reward causes the agent to learn a different policy
- Even a small misaligned reward (0.5) can distort behavior significantly
- This illustrates the challenge of reward specification in real applications
- This is why RLHF and reward modeling are critical for aligning learned behavior with human intent

**Part 4 (Discount Factor):**
- Low gamma (0.5) makes the agent myopic: only nearby rewards matter
- High gamma (0.99) makes the agent far-sighted: distant rewards matter equally
- The start state (0, 0) has similar value under all gammas, but values diverge in the middle
- This shows how gamma controls the temporal horizon of planning